# PharmaMind ML Model Exploration

This notebook provides an academic exploration of the synthetic data generation, feature engineering, and model training processes used in the PharmaMind backend. 

## 1. Introduction & Objectives
PharmaMind utilizes 7 machine learning models to forecast operational bottlenecks:
- Shortage Predictor
- Acceptance Predictor
- Demand Forecaster
- Trust Risk Predictor
- Workforce Health Regressor
- Retention Predictor
- Closure Risk Predictor

Because real-world pharmacy staffing data is proprietary and sensitive, `seed.py` employs synthetic data generation bounded by realistic operational constraints.

## 2. Temporal Feature Engineering (Cyclic Encoding)
Months (1-12) and Days of the Week (0-6) are cyclical. Using raw integers leads models to incorrectly assume that December (12) and January (1) are numerically distant.

We employ sine and cosine transformations to map these onto a circle:

$$ \sin\left(\frac{2\pi x}{\max(x)}\right), \cos\left(\frac{2\pi x}{\max(x)}\right) $$

In [ ]:
import sys
import os

# Add the backend folder to the Python path so we can import from it
sys.path.insert(0, os.path.abspath('../backend'))


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def cyclic_encode_month(month):
    rad = 2 * math.pi * (month - 1) / 12
    return math.sin(rad), math.cos(rad)

months = list(range(1, 13))
encoded = [cyclic_encode_month(m) for m in months]
sin_m, cos_m = zip(*encoded)

plt.figure(figsize=(5,5))
plt.scatter(sin_m, cos_m, c=months, cmap='hsv', s=100)
for i, m in enumerate(months):
    plt.annotate(str(m), (sin_m[i]*1.1, cos_m[i]*1.1))
plt.title('Cyclic Encoding of Months')
plt.axis('equal')
plt.show()

## 3. Justification of Closure Risk Threshold
The `closure_risk_label` is derived by thresholding a continuous operational risk score at `0.35` (35%). 
This was determined via heuristics: a pharmacy running with a 35% drop in workforce health combined with active unfilled shifts historically requires emergency float coverage within 48 hours to remain legally compliant for dispensing.